# 03 — Feature Engineering (v3)
**What changed from v2:**
- `dropna()` is now surgical — only drops rows where the TARGET column is missing,
  not where any feature is missing. Features with NaN get filled with 0 at model time.
- Added date-range diagnostic after loading to catch truncated CSVs early.
- SVI weekly-to-daily reindex uses `limit=4` to avoid propagating stale values
  across long holiday gaps but does NOT drop rows just because SVI is NaN.
- All bounded series (FG_Index, RSI etc.) exempt from ADF differencing as before.

Reads from `data/clean/` — writes to `data/features/`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
warnings.filterwarnings('ignore')

with open('/content/drive/MyDrive/CapstoneDA/config.json') as f:
    CFG = json.load(f)

CLEAN    = CFG['PATHS']['clean']
FEATURES = CFG['PATHS']['features']
PLOTS    = CFG['PATHS']['plots']
SEED     = CFG['SEED']
np.random.seed(SEED)

import os
os.makedirs(FEATURES, exist_ok=True)

print('Config loaded.')
print(f'Study period : {CFG["START_DATE"]} to {CFG["END_DATE"]}')
print(f'Reading from : {CLEAN}')
print(f'Writing to   : {FEATURES}')

Mounted at /content/drive
Config loaded.
Study period : 2016-01-01 to 2025-12-31
Reading from : /content/drive/MyDrive/CapstoneDA/data/clean
Writing to   : /content/drive/MyDrive/CapstoneDA/data/features


In [2]:
def load(path, label):
    df = pd.read_csv(path, index_col='Date', parse_dates=['Date'])
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        df.index = df.index.tz_convert(None)
    else:
        df.index = pd.DatetimeIndex(df.index)
    print(f'  {label:35} {len(df):5} rows  '
          f'{df.index.min().date()} to {df.index.max().date()}')
    return df

print('Loading clean files:')
sp500      = load(f'{CLEAN}/sp500_clean.csv',        'S&P 500')
nifty      = load(f'{CLEAN}/nifty_clean.csv',        'NIFTY 50')
macro      = load(f'{CLEAN}/macro_clean.csv',        'Macro')
fg         = load(f'{CLEAN}/fear_greed_clean.csv',   'Fear & Greed')
svi_sp500  = load(f'{CLEAN}/svi_sp500_daily.csv',   'SVI SP500')
svi_nifty  = load(f'{CLEAN}/svi_nifty_daily.csv',   'SVI NIFTY')
nifty_lags = load(f'{CLEAN}/nifty_lag_features.csv','NIFTY lags')

# Sanity check — warn if any source ends before the expected end date
expected_end = pd.Timestamp(CFG['END_DATE'])
print()
print('Date range check:')
for name, df in [('SP500', sp500), ('NIFTY', nifty), ('Macro', macro),
                 ('FG', fg), ('SVI_SP500', svi_sp500), ('SVI_NIFTY', svi_nifty)]:
    gap = (expected_end - df.index.max()).days
    flag = f'  WARNING: ends {gap} days early' if gap > 30 else 'OK'
    print(f'  {name:12} ends {df.index.max().date()}  {flag}')

Loading clean files:
  S&P 500                              2513 rows  2016-01-04 to 2025-12-30
  NIFTY 50                             2463 rows  2016-01-04 to 2025-12-30
  Macro                                2608 rows  2016-01-04 to 2025-12-31
  Fear & Greed                         2609 rows  2016-01-01 to 2025-12-31
  SVI SP500                            2513 rows  2016-01-04 to 2025-12-30
  SVI NIFTY                            2463 rows  2016-01-04 to 2025-12-30
  NIFTY lags                           2609 rows  2016-01-01 to 2025-12-31

Date range check:
  SP500        ends 2025-12-30  OK
  NIFTY        ends 2025-12-30  OK
  Macro        ends 2025-12-31  OK
  FG           ends 2025-12-31  OK
  SVI_SP500    ends 2025-12-30  OK
  SVI_NIFTY    ends 2025-12-30  OK


In [3]:
# Target variable
sp500['Target_Dir'] = (sp500['close'].shift(-1) > sp500['close']).astype(int)
nifty['Target_Dir'] = (nifty['close'].shift(-1) > nifty['close']).astype(int)

# Drop only the LAST row (no next-day close to label) — not any other rows
sp500 = sp500.iloc[:-1].copy()
nifty = nifty.iloc[:-1].copy()

print(f'S&P 500  : {len(sp500)} rows after target creation  '
      f'({sp500.index.min().date()} to {sp500.index.max().date()})')
print(f'NIFTY 50 : {len(nifty)} rows after target creation  '
      f'({nifty.index.min().date()} to {nifty.index.max().date()})')
print(f'Up-day rate: SP500={sp500["Target_Dir"].mean():.1%}  '
      f'NIFTY={nifty["Target_Dir"].mean():.1%}')

S&P 500  : 2512 rows after target creation  (2016-01-04 to 2025-12-29)
NIFTY 50 : 2462 rows after target creation  (2016-01-04 to 2025-12-29)
Up-day rate: SP500=54.8%  NIFTY=54.2%


In [4]:
# Technical indicators

def rsi(close, window=14):
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(span=window, min_periods=window).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=window, min_periods=window).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

def macd_signal(close, fast=12, slow=26, sig=9):
    line   = close.ewm(span=fast,min_periods=fast).mean() - \
             close.ewm(span=slow,min_periods=slow).mean()
    return line - line.ewm(span=sig, min_periods=sig).mean()

def atr(high, low, close, w=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(span=w, min_periods=w).mean()

def rolling_zscore(s, window=30, min_periods=10):
    m  = s.rolling(window, min_periods=min_periods).mean()
    sd = s.rolling(window, min_periods=min_periods).std().replace(0, np.nan)
    return (s - m) / sd

def add_technicals(df):
    c = df['close']
    df['Log_Ret_t']  = np.log(c / c.shift(1))
    df['Log_Ret_1']  = df['Log_Ret_t'].shift(1)
    df['Log_Ret_2']  = df['Log_Ret_t'].shift(2)
    df['Log_Ret_5']  = df['Log_Ret_t'].shift(5)
    df['Ret_Mean_5']  = df['Log_Ret_t'].rolling(5,  min_periods=3).mean()
    df['Ret_Mean_10'] = df['Log_Ret_t'].rolling(10, min_periods=5).mean()
    df['Ret_Mean_20'] = df['Log_Ret_t'].rolling(20, min_periods=10).mean()
    df['Ret_Std_5']   = df['Log_Ret_t'].rolling(5,  min_periods=3).std()
    df['Ret_Std_10']  = df['Log_Ret_t'].rolling(10, min_periods=5).std()
    df['RSI_14']      = rsi(c, 14)
    df['RSI_5']       = rsi(c, 5)
    df['MACD_Signal'] = macd_signal(c)
    df['MACD_Hist']   = macd_signal(c)   # same formula, kept separate
    df['ATR_14']      = atr(df['high'], df['low'], c)
    ma50              = c.rolling(50,  min_periods=30).mean()
    ma200             = c.rolling(200, min_periods=100).mean()
    df['Above_MA50']  = (c > ma50).astype(float)
    df['Above_MA200'] = (c > ma200).astype(float)
    df['Price_MA50']  = (c - ma50) / ma50
    return df

sp500 = add_technicals(sp500)
nifty = add_technicals(nifty)
print('Technical features done.')

Technical features done.


In [5]:
# Macro features

def add_macro(df, macro_df, include_usdinr=False):
    m = macro_df.reindex(df.index, method='ffill')
    df['VIX_Change']    = m['VIX'].diff()
    df['VIX_Level']     = rolling_zscore(m['VIX'], 60)
    df['VIX_Regime']    = (m['VIX'] > m['VIX'].rolling(90, min_periods=40).mean()).astype(float)
    raw_spread          = m['US10Y'] - m['US2Y']
    df['Term_Spread']   = raw_spread.rolling(5, min_periods=2).mean()
    df['Oil_Ret']       = np.log(m['BRENT'] / m['BRENT'].shift(1))
    df['Gold_Ret']      = np.log(m['GOLD']  / m['GOLD'].shift(1))
    df['Gold_Oil_Ratio']= df['Gold_Ret'] - df['Oil_Ret']
    if include_usdinr and 'USDINR' in m.columns:
        df['USDINR_Ret'] = np.log(m['USDINR'] / m['USDINR'].shift(1))
    return df

sp500 = add_macro(sp500, macro, include_usdinr=False)
nifty = add_macro(nifty, macro, include_usdinr=True)
print('Macro features done.')

Macro features done.


In [6]:
# Fear & Greed features
# FG_Index is bounded 0-100 — use rolling z-score, NOT differencing.

def add_fg(df, fg_df):
    fg = fg_df.reindex(df.index, method='ffill')
    df['FG_Level']   = rolling_zscore(fg['FG_Index'], 30)
    df['FG_Delta']   = fg['FG_Index'].diff()
    df['FG_Zone']    = fg['FG_Zone'].astype(float)
    df['FG_Extreme'] = ((fg['FG_Index'] < 25) | (fg['FG_Index'] > 75)).astype(float)
    df['FG_MA5']     = rolling_zscore(fg['FG_Index'].rolling(5, min_periods=3).mean(), 30)
    return df

sp500 = add_fg(sp500, fg)
nifty = add_fg(nifty, fg)
print('FG features done.')

FG features done.


In [7]:
# SVI features
# SVI comes in as weekly data expanded to daily via forward-fill in NB02.
# We apply rolling z-score here. NaN values in SVI do NOT cause row drops —
# they will be filled with 0 at model time.

def add_svi(df, svi_df, market_kw, index_kw):
    svi = svi_df.reindex(df.index, method='ffill', limit=4)

    for name, kw in [('SVI_Market', market_kw), ('SVI_Index', index_kw)]:
        col = svi.get(kw) if hasattr(svi, 'get') else (
            svi[kw] if kw in svi.columns else svi.mean(axis=1)
        )
        if kw in svi.columns:
            col = svi[kw]
        else:
            col = svi.mean(axis=1)
        m    = col.rolling(30, min_periods=10).mean()
        sd   = col.rolling(30, min_periods=10).std().replace(0, np.nan)
        df[name] = (col - m) / sd

    df['SVI_Mom'] = df['SVI_Market'] - df['SVI_Market'].shift(20)
    return df

sp500 = add_svi(sp500, svi_sp500,
               CFG['SVI_KEYWORDS_SP500'][0], CFG['SVI_KEYWORDS_SP500'][1])
nifty = add_svi(nifty, svi_nifty,
               CFG['SVI_KEYWORDS_NIFTY'][0], CFG['SVI_KEYWORDS_NIFTY'][1])
print('SVI features done.')

SVI features done.


In [8]:
# NIFTY lag features (Global Clock Protocol)

lag = nifty_lags.reindex(nifty.index, method='ffill')
nifty['SP500_t1_Ret'] = lag['SP500_t1_Ret']

raw_fg_t1              = lag['FG_t1_Index']
m                      = raw_fg_t1.rolling(30, min_periods=10).mean()
sd                     = raw_fg_t1.rolling(30, min_periods=10).std().replace(0, np.nan)
nifty['FG_t1_Level']   = (raw_fg_t1 - m) / sd
nifty['FG_t1_Delta']   = raw_fg_t1.diff()

print('NIFTY lag features done.')

NIFTY lag features done.


In [9]:
# Interaction features

def add_interactions(df):
    if 'FG_Level' in df.columns and 'SVI_Market' in df.columns:
        df['FG_x_SVI']     = df['FG_Level'] * df['SVI_Market']
    if 'Log_Ret_t' in df.columns and 'ATR_14' in df.columns:
        df['Vol_Adj_Ret']  = df['Log_Ret_t'] / df['ATR_14'].replace(0, np.nan)
    if 'VIX_Change' in df.columns and 'Term_Spread' in df.columns:
        df['VIX_x_Spread'] = df['VIX_Change'] * df['Term_Spread']
    return df

sp500 = add_interactions(sp500)
nifty = add_interactions(nifty)
print('Interaction features done.')

Interaction features done.


In [10]:
# Define feature lists

BASE_TECH  = ['Log_Ret_t','Log_Ret_1','Log_Ret_2','Log_Ret_5',
              'Ret_Mean_5','Ret_Mean_10','Ret_Mean_20','Ret_Std_5','Ret_Std_10',
              'RSI_14','RSI_5','MACD_Signal','MACD_Hist','ATR_14',
              'Above_MA50','Above_MA200','Price_MA50']

BASE_MACRO = ['VIX_Change','VIX_Level','VIX_Regime','Term_Spread',
              'Oil_Ret','Gold_Ret','Gold_Oil_Ratio']

BASE_FG    = ['FG_Level','FG_Delta','FG_Zone','FG_Extreme','FG_MA5']

BASE_SVI   = ['SVI_Market','SVI_Index','SVI_Mom']

BASE_INTER = ['FG_x_SVI','Vol_Adj_Ret','VIX_x_Spread']

ALL_SP500  = BASE_TECH + BASE_MACRO + BASE_FG + BASE_SVI + BASE_INTER
ALL_NIFTY  = ALL_SP500 + ['USDINR_Ret','SP500_t1_Ret','FG_t1_Level','FG_t1_Delta']

sp500_feat_cols = [f for f in ALL_SP500 if f in sp500.columns]
nifty_feat_cols = [f for f in ALL_NIFTY  if f in nifty.columns]

print(f'S&P 500 candidate features : {len(sp500_feat_cols)}')
print(f'NIFTY 50 candidate features: {len(nifty_feat_cols)}')

S&P 500 candidate features : 35
NIFTY 50 candidate features: 39


In [ ]:
# ADF stationarity audit
# Bounded / mean-reverting series are exempt from differencing.

ADF_EXEMPT = {
    'RSI_14','RSI_5','FG_Zone','FG_Extreme','FG_Level','FG_MA5',
    'FG_t1_Level','Above_MA50','Above_MA200','VIX_Regime',
    'SVI_Market','SVI_Index','SVI_Mom','FG_x_SVI','Vol_Adj_Ret'
}

def adf_audit(df, feature_cols, label):
    results = []
    for col in feature_cols:
        if col not in df.columns:
            continue
        s = df[col].dropna()
        if len(s) < 30:
            results.append({'Feature':col,'p':None,'Action':'skip (too short)'})
            continue
        try:
            _, p, *_ = adfuller(s, autolag='AIC')
            exempt   = col in ADF_EXEMPT
            action   = 'EXEMPT' if exempt else ('PASS' if p < 0.05 else 'DIFFERENCE')
            results.append({'Feature':col,'p':round(p,4),'Action':action})
        except Exception:
            results.append({'Feature':col,'p':None,'Action':'error'})
    rpt = pd.DataFrame(results)
    print(f'\n{label}')
    print(rpt.to_string(index=False))
    n_diff = (rpt['Action'] == 'DIFFERENCE').sum()
    print(f'  {n_diff} feature(s) to be differenced')
    return rpt

adf_sp = adf_audit(sp500, sp500_feat_cols, 'ADF — S&P 500')
adf_ni = adf_audit(nifty, nifty_feat_cols, 'ADF — NIFTY 50')

In [ ]:
# Apply differencing only where ADF says DIFFERENCE (not exempt)

def apply_diff(df, adf_report):
    df = df.copy()
    for _, row in adf_report.iterrows():
        if row['Action'] == 'DIFFERENCE' and row['Feature'] in df.columns:
            df[row['Feature']] = df[row['Feature']].diff()
    return df   # DO NOT dropna() here

sp500_feat = apply_diff(sp500, adf_sp)
nifty_feat = apply_diff(nifty, adf_ni)

# Drop only rows where Target_Dir is missing (the very last row)
# Every other NaN will be filled with 0 at model time via .fillna(0)
sp500_feat = sp500_feat.dropna(subset=['Target_Dir'])
nifty_feat = nifty_feat.dropna(subset=['Target_Dir'])

sp500_feat_cols = [f for f in sp500_feat_cols if f in sp500_feat.columns]
nifty_feat_cols = [f for f in nifty_feat_cols if f in nifty_feat.columns]

print(f'S&P 500 final: {sp500_feat.shape}  '
      f'{sp500_feat.index.min().date()} to {sp500_feat.index.max().date()}')
print(f'NIFTY 50 final: {nifty_feat.shape}  '
      f'{nifty_feat.index.min().date()} to {nifty_feat.index.max().date()}')

# Report how many NaN values remain per feature
# These are fine — they will be filled at model time
sp_nan = sp500_feat[sp500_feat_cols].isnull().sum()
sp_nan = sp_nan[sp_nan > 0]
if not sp_nan.empty:
    print(f'\nS&P 500 remaining NaN counts (filled with 0 at model time):')
    print(sp_nan.to_string())

In [ ]:
# Save feature matrices and registry

sp500_feat.to_csv(f'{FEATURES}/sp500_features.csv')
nifty_feat.to_csv(f'{FEATURES}/nifty_features.csv')

registry = {
    'sp500_all_features' : sp500_feat_cols,
    'nifty_all_features' : nifty_feat_cols,
    'baseline_features'  : BASE_TECH + BASE_MACRO,
    'alt_data_features'  : BASE_FG + BASE_SVI,
    'feature_families'   : {
        'technical'   : BASE_TECH,
        'macro'       : BASE_MACRO,
        'fear_greed'  : BASE_FG,
        'svi'         : BASE_SVI,
        'interaction' : BASE_INTER,
    },
    'target_column' : 'Target_Dir',
    'adf_exempt'    : list(ADF_EXEMPT),
    'nifty_only'    : ['USDINR_Ret','SP500_t1_Ret','FG_t1_Level','FG_t1_Delta'],
}
with open(f'{FEATURES}/feature_registry.json','w') as f:
    json.dump(registry, f, indent=2)

adf_sp.to_csv(f'{FEATURES}/adf_audit_sp500.csv', index=False)
adf_ni.to_csv(f'{FEATURES}/adf_audit_nifty.csv', index=False)

print('Saved to data/features/')
print(f'  sp500_features.csv  {sp500_feat.shape}')
print(f'  nifty_features.csv  {nifty_feat.shape}')
print(f'  feature_registry.json')

In [ ]:
# Correlation heatmap

fig, axes = plt.subplots(1, 2, figsize=(20, 14))
for ax, feat_df, feat_cols, label in [
    (axes[0], sp500_feat, sp500_feat_cols, 'S&P 500'),
    (axes[1], nifty_feat, nifty_feat_cols, 'NIFTY 50'),
]:
    cols = [f for f in feat_cols if f in feat_df.columns] + ['Target_Dir']
    corr = feat_df[cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, linewidths=0.3,
                annot_kws={'size': 6}, ax=ax, cbar_kws={'shrink': 0.6})
    ax.set_title(label, fontsize=12)
plt.tight_layout()
plt.savefig(f'{PLOTS}/03_correlation_heatmap_v3.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlations with target
print('Top 15 correlations with Target_Dir (S&P 500):')
tc = sp500_feat[sp500_feat_cols + ['Target_Dir']].corr()['Target_Dir'].drop('Target_Dir')
for feat in tc.abs().sort_values(ascending=False).index[:15]:
    val = tc[feat]
    bar = '#' * int(abs(val) * 200)
    print(f'  {feat:22}  {val:+.4f}  {bar}')